# Check concatenated results from Gelato

- author : Sylvie Dagoret-Campagne
- creation date : 2024-08-29
- update : 2024-05-29
- last update : 2024-08-31

In [ ]:
import pandas as pd
from astropy.io import fits
from astropy.table import Table
import re
import os
import gelato
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
import matplotlib as mpl
mpl.rcParams['font.size'] = 16

## Config

In [ ]:
version = "v3"
path_params = f"./ExampleParametersFitInNb_{version}.json"

snr_min = 0.1

In [ ]:
# Create Parameters dictionary
params_gel = gelato.ConstructParams.construct(path_params)

In [ ]:
file_fit_results_fn = "GELATO-results.fits"

In [ ]:
OutFolder = params_gel["OutFolder"]

In [ ]:
file_fit_results_fullfn=os.path.join(OutFolder,file_fit_results_fn)

In [ ]:
flag_save_filelist = False

## Read full object list

In [ ]:
t = Table.read(file_fit_results_fullfn)

In [ ]:
df = t.to_pandas()

In [ ]:
for col in df.columns:
    #if "Flux_err" in col:
    print(col)

## Plots

### Chi2

In [ ]:
the_chi2_values = list(df.rChi2.values)

In [ ]:
title = "GETATO Fit (Fors2) Reduced Chi2"
xlabel = "$\chi^2/Ndf$"
fig, ax = plt.subplots(figsize=(6, 4), facecolor='white',
                       layout='constrained')
ax.hist(the_chi2_values ,bins=100,range=(0,5.));
ax.grid()
ax.set_title(title)
ax.set_xlabel(xlabel)

## Emission Lines

In [ ]:
groups = params_gel['EmissionGroups']

In [ ]:
for group in groups:
    g_name = group['Name']
    for species in group['Species']:
        s_name = species['Name']
        all_lines = species['Lines']
        for line in all_lines:
            wl = line['Wavelength']
            l_tag =  g_name +'_' + s_name +'_' + str(wl)
            print(l_tag)
            if g_name == "AGN" and s_name == "[OIII]":
                c_tag =  'Outflow' +'_' + s_name + '_Outflow_' + str(wl)
                print(c_tag)
            if g_name == "Balmer" and s_name == "HI":
                c_tag =  g_name +'_' + s_name + '_Broad_' + str(wl)
                print(c_tag)
            

#### Flux in FLAM

In [ ]:
def PlotFlux(data,title,xlabel):
    fig, ax = plt.subplots(figsize=(5, 4), facecolor='white',layout='constrained')
    nbins = 100
    
    #ax.hist(data ,bins=nbins);
    
    hist, bins = np.histogram(data, bins=nbins)
    logbinmin = np.log10(bins[0])
    logbinmax = np.log10(bins[-1])
    if np.isnan(logbinmin)  or np.isnan(logbinmax):
        # plot on linear scale
        ax.hist(data, bins=nbins)
    else:
        logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))
        ax.hist(data, bins=logbins)
        plt.gca()
        plt.xscale('log')
    
    
    ax.grid()
    ax.set_title(title)
    ax.set_xlabel(xlabel)

    plt.show()

##### Selection

In [ ]:
cut = df.rChi2.values< 2.0

In [ ]:
for group in groups:
    g_name = group['Name']
    for species in group['Species']:
        s_name = species['Name']
        all_lines = species['Lines']
        for line in all_lines:
            wl = line['Wavelength']
            c_tag =  g_name +'_' + s_name +'_' + str(wl)
            c_name = c_tag +  "_Flux"
            c_name_err = c_tag +  "_Flux_err"
            values = df[c_name][cut].values
            values_err = df[c_name_err][cut].values
            s_n = values/values_err
            cut_2 = (np.abs(s_n)> snr_min) & ( ~np.isnan(values) & (values !=0))
            title = c_tag
            xlabel = c_name
            val_to_plot = values[cut_2]
            if len(val_to_plot)>0:
                PlotFlux(val_to_plot,title,xlabel)
 
            if g_name == "AGN" and s_name == "[OIII]":
                c_tag =  'Outflow' +'_' + s_name + '_Outflow_' + str(wl)
                c_name = c_tag +  "_Flux"
                c_name_err = c_tag +  "_Flux_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> snr_min) & (~np.isnan(values) & (values !=0.0))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)
                
            if g_name == "Balmer" and s_name == "HI":
                c_tag =  g_name +'_' + s_name + '_Broad_' + str(wl)
                c_name = c_tag +  "_Flux"
                c_name_err = c_tag +  "_Flux_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> snr_min) & (~np.isnan(values) & (values !=0.0 ))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)

## RAmp

In [ ]:
for group in groups:
    g_name = group['Name']
    for species in group['Species']:
        s_name = species['Name']
        all_lines = species['Lines']
        for line in all_lines:
            wl = line['Wavelength']
            c_tag =  g_name +'_' + s_name +'_' + str(wl)
            c_name = c_tag +  "_RAmp"
            c_name_err = c_tag +  "_RAmp_err"
            values = df[c_name][cut].values
            values_err = df[c_name_err][cut].values
            s_n = values/values_err
            cut_2 = (np.abs(s_n)> snr_min) & ( ~np.isnan(values) & (values !=0))
            title = c_tag
            xlabel = c_name
            val_to_plot = values[cut_2]
            if len(val_to_plot)>0:
                PlotFlux(val_to_plot,title,xlabel)
 
            if g_name == "AGN" and s_name == "[OIII]":
                c_tag =  'Outflow' +'_' + s_name + '_Outflow_' + str(wl)
                c_name = c_tag +  "_REW"
                c_name_err = c_tag +  "_REW_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> snr_min) & (~np.isnan(values) & (values !=0.0))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)
                
            if g_name == "Balmer" and s_name == "HI":
                c_tag =  g_name +'_' + s_name + '_Broad_' + str(wl)
                c_name = c_tag +  "_REW"
                c_name_err = c_tag +  "_REW_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> snr_min) & (~np.isnan(values) & (values !=0.0 ))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)

#### EQW

In [ ]:
for group in groups:
    g_name = group['Name']
    for species in group['Species']:
        s_name = species['Name']
        all_lines = species['Lines']
        for line in all_lines:
            wl = line['Wavelength']
            c_tag =  g_name +'_' + s_name +'_' + str(wl)
            c_name = c_tag +  "_REW"
            c_name_err = c_tag +  "_REW_err"
            values = df[c_name][cut].values
            values_err = df[c_name_err][cut].values
            s_n = values/values_err
            cut_2 = (np.abs(s_n)> snr_min) & ( ~np.isnan(values) & (values !=0))
            title = c_tag
            xlabel = c_name
            val_to_plot = values[cut_2]
            if len(val_to_plot)>0:
                PlotFlux(val_to_plot,title,xlabel)
 
            if g_name == "AGN" and s_name == "[OIII]":
                c_tag =  'Outflow' +'_' + s_name + '_Outflow_' + str(wl)
                c_name = c_tag +  "_REW"
                c_name_err = c_tag +  "_REW_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> snr_min) & (~np.isnan(values) & (values !=0.0))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)
                
            if g_name == "Balmer" and s_name == "HI":
                c_tag =  g_name +'_' + s_name + '_Broad_' + str(wl)
                c_name = c_tag +  "_REW"
                c_name_err = c_tag +  "_REW_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> snr_min) & (~np.isnan(values) & (values !=0.0 ))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)

## Selection

In [ ]:
df_sel = df[cut]

## Correlations

In [ ]:
def PlotCorr(data1,data2,title,label1,label2):
    fig, ax = plt.subplots(figsize=(5, 5), facecolor='white',layout='constrained')   
    ax.scatter(data1,data2, s=50, c="r", marker='o')
    ax.set_title(title)
    ax.set_xlabel(label1)
    ax.set_ylabel(label2)
    ax.grid()

    plt.show()

In [ ]:
data2 = df_sel['Balmer_HI_Broad_4341.68_Flux']
data1 = df_sel['Balmer_HI_4341.68_Flux']

In [ ]:
data1 = data1[~data2.isna()].values
data2 =data2[~data2.isna()].values

In [ ]:
title = "Component Balmer_HI_Broad_4341.68_Flux vs \nEmis line Balmer_HI_4341.68_Flux"
label1= 'Balmer_HI_4341.68_Flux'
label2 = 'Balmer_HI_Broad_4341.68_Flux'
PlotCorr(data1,data2,title,label1,label2)

In [ ]:
data2 = df_sel['Balmer_HI_Broad_4862.68_Flux']
data1 = df_sel['Balmer_HI_4862.68_Flux']
data1 = data1[~data2.isna()].values
data2 =data2[~data2.isna()].values

In [ ]:
title = "Component Balmer_HI_Broad_4862.68_Flux vs \nEmis line Balmer_HI_4862.68_Flux"
label1= 'Balmer_HI_4862.68_Flux'
label2 = 'Balmer_HI_Broad_4862.68_Flux'
PlotCorr(data1,data2,title,label1,label2)

In [ ]:
data2 = df_sel['Balmer_HI_Broad_6564.61_Flux']
data1 = df_sel['Balmer_HI_6564.61_Flux']
data1 = data1[~data2.isna()].values
data2 =data2[~data2.isna()].values

In [ ]:
title = "Component Balmer_HI_Broad_6564.61_Flux vs \nEmis line Balmer_HI_6564.61_Flux"
label1= 'Balmer_HI_6564.61_Flux'
label2 = 'Balmer_HI_Broad_6564.61_Flux'
PlotCorr(data1,data2,title,label1,label2)

In [ ]:
data2 = df_sel['Outflow_[OIII]_Outflow_4364.436_Flux']
data1 = df_sel['AGN_[OIII]_4364.436_Flux']
data1 = data1[~data2.isna()].values
data2 =data2[~data2.isna()].values

In [ ]:
title = "Component Outflow_[OIII]_Outflow_4364.436_Flux vs \nEmis line AGN_[OIII]_4364.436_Flux"
label1= 'AGN_[OIII]_4364.436_Flux'
label2 = 'Outflow_[OIII]_Outflow_4364.436_Flux'
PlotCorr(data1,data2,title,label1,label2)

In [ ]:
data2 = df_sel['Outflow_[OIII]_Outflow_4960.295_Flux']
data1 = df_sel['AGN_[OIII]_4960.295_Flux']
data1 = data1[~data2.isna()].values
data2 =data2[~data2.isna()].values

In [ ]:
title = "Component Outflow_[OIII]_Outflow_4960.295_Flux vs \nEmis line AGN_[OIII]_4960.295_Flux"
label1= 'AGN_[OIII]_4960.295_Flux'
label2 = 'Outflow_[OIII]_Outflow_4960.295_Flux'
PlotCorr(data1,data2,title,label1,label2)

In [ ]:
data2 = df_sel['Outflow_[OIII]_Outflow_5008.24_Flux']
data1 = df_sel['AGN_[OIII]_5008.24_Flux']
data1 = data1[~data2.isna()].values
data2 =data2[~data2.isna()].values

In [ ]:
title = "Component Outflow_[OIII]_Outflow_5008.24_Flux vs \nEmis line AGN_[OIII]_5008.24_Flux"
label1= 'AGN_[OIII]_5008.24_Flux'
label2 = 'Outflow_[OIII]_Outflow_5008.24_Flux'
PlotCorr(data1,data2,title,label1,label2)

## BPT Plots

### SF : 2 species (OI, OII)

- SF_[OI]_6302.046
- SF_[OI]_6365.536

- ........................
    
- SF_[OII]_3728.48

### AGN : 5 species (SII, NII, OIII, NeIII, NeV)

- AGN_[SII]_6718.29
- AGN_[SII]_6732.67

- ........................
 
- AGN_[NII]_6585.27
- AGN_[NII]_6549.86

- ........................

- AGN_[NeIII]_3869.86

- ........................
  
- AGN_[NeV]_3346.79
- AGN_[NeV]_3426.85


- ........................
  
- AGN_[OIII]_5008.24
- AGN_[OIII]_4960.295
- AGN_[OIII]_4364.436

### Balmer : 1 specie : (HI)

- Balmer_HI_6564.61
- Balmer_HI_4862.68
- Balmer_HI_4341.68

### Component Balmer_Broad

- Balmer_HI_Broad_6564.61
- Balmer_HI_Broad_4862.68
- Balmer_HI_Broad_4341.68

### Component Outflow_Outflow

- Outflow_[OIII]_Outflow_4960.295
- Outflow_[OIII]_Outflow_5008.24 
- Outflow_[OIII]_Outflow_4364.436

### Types of BTP plots

- $([OIII]/H_\beta) \, vs \, ([NII]/H_\alpha)$

- $([OIII]/H_\beta) \, vs \, ([SII]/H_\alpha)$
 
- $([OIII]/H_\beta) \, vs \, ([OI]/H_\alpha)$

In [ ]:
#OIII
oiii_1 = df_sel['AGN_[OIII]_5008.24_Flux']
oiii_2 = df_sel['AGN_[OIII]_4960.295_Flux']
#oiii_3 = t['AGN_[OIII]_4364.436_Flux']

#NII
nii_1 = df_sel['AGN_[NII]_6585.27_Flux']
nii_2 = df_sel['AGN_[NII]_6549.86_Flux']

#SII
sii_1 = df_sel['AGN_[SII]_6718.29_Flux']
sii_2 = df_sel['AGN_[SII]_6732.67_Flux']

#OI

oi1 = df_sel['SF_[OI]_6302.046_Flux']
oi2 = df_sel['SF_[OI]_6365.536_Flux']

# Halpha, Hbeta
df_sel["ha"] = df_sel['Balmer_HI_6564.61_Flux']
df_sel["hb"] = df_sel['Balmer_HI_4862.68_Flux']
df_sel["ha_err"] = df_sel['Balmer_HI_6564.61_Flux_err']
df_sel["hb_err"] = df_sel['Balmer_HI_4862.68_Flux_err']


In [ ]:
def sum_oiii(row):
    flux1 = row['AGN_[OIII]_5008.24_Flux']
    flux1_err = row['AGN_[OIII]_5008.24_Flux_err']
    sn1 = flux1/flux1_err
    
    flux2 = row['AGN_[OIII]_4960.295_Flux']
    flux2_err = row['AGN_[OIII]_4960.295_Flux_err']
    sn2 = flux2/flux2_err


    flux_sum = 0.0
    flux_err_2 = 0.0

    if (not np.isnan(sn1)) and (sn1>snr_min):
        flux_sum += flux1
        flux_err_2 += flux1_err**2
        
    if (not np.isnan(sn2)) and (sn2>snr_min):
        flux_sum += flux2
        flux_err_2 += flux2_err**2
        
    if flux_sum >0:
        return flux_sum,np.sqrt(flux_err_2)
    else:
        return np.nan,np.nan

In [ ]:
def sum_nii(row):
    flux1 = row['AGN_[NII]_6585.27_Flux']
    flux1_err = row['AGN_[NII]_6585.27_Flux_err']
    sn1 = flux1/flux1_err
    
    flux2 = row['AGN_[NII]_6549.86_Flux']
    flux2_err = row['AGN_[NII]_6549.86_Flux_err']
    sn2 = flux2/flux2_err


    flux_sum = 0.0
    flux_err_2 = 0.0

    if (not np.isnan(sn1)) and (sn1>snr_min):
        flux_sum += flux1
        flux_err_2 += flux1_err**2
        
    if (not np.isnan(sn2)) and (sn2>snr_min):
        flux_sum += flux2
        flux_err_2 += flux2_err**2
        
    if flux_sum >0:
        return flux_sum,np.sqrt(flux_err_2)
    else:
        return np.nan,np.nan

In [ ]:
def sum_sii(row):
    
    flux1 = row['AGN_[SII]_6718.29_Flux']
    flux1_err = row['AGN_[SII]_6718.29_Flux_err']
    sn1 = flux1/flux1_err
    
    flux2 = row['AGN_[SII]_6732.67_Flux']
    flux2_err = row['AGN_[SII]_6732.67_Flux_err']
    sn2 = flux2/flux2_err


    flux_sum = 0.0
    flux_err_2 = 0.0

    if (not np.isnan(sn1)) and (sn1>snr_min):
        flux_sum += flux1
        flux_err_2 += flux1_err**2
        
    if (not np.isnan(sn2)) and (sn2>snr_min):
        flux_sum += flux2
        flux_err_2 += flux2_err**2
        
    if flux_sum >0:
        return flux_sum,np.sqrt(flux_err_2)
    else:
        return np.nan,np.nan

In [ ]:
def sum_oi(row):
    
    flux1 = row['SF_[OI]_6302.046_Flux']
    flux1_err = row['SF_[OI]_6302.046_Flux_err']
    sn1 = flux1/flux1_err
    
    flux2 = row['SF_[OI]_6365.536_Flux']
    flux2_err = row['SF_[OI]_6365.536_Flux_err']
    sn2 = flux2/flux2_err


    flux_sum = 0.0
    flux_err_2 = 0.0

    if (not np.isnan(sn1)) and (sn1>snr_min):
        flux_sum += flux1
        flux_err_2 += flux1_err**2
        
    if (not np.isnan(sn2)) and (sn2>snr_min):
        flux_sum += flux2
        flux_err_2 += flux2_err**2
        
    if flux_sum >0:
        return flux_sum,np.sqrt(flux_err_2)
    else:
        return np.nan,np.nan

In [ ]:
# apply functions above (flux sum in different emission lines)
df_sel[["oiii","oiii_err"]]=df_sel.apply(sum_oiii,axis=1,result_type="expand")
df_sel[["nii","nii_err"]]=df_sel.apply(sum_nii,axis=1,result_type="expand")
df_sel[["sii","sii_err"]]=df_sel.apply(sum_sii,axis=1,result_type="expand")
df_sel[["oi","oi_err"]]=df_sel.apply(sum_oi,axis=1,result_type="expand")

In [ ]:
# Check
#df_sel["oiii"].dropna()
#df_sel["nii"].dropna()
#df_sel["sii"].dropna()
#df_sel["oi"].dropna()

In [ ]:
# BPT ordinate
df_sel["oiii_hb"] = df_sel["oiii"]/df_sel["hb"]
df_sel["oiii_hb_err"] = df_sel["oiii_hb"]*np.sqrt( (df_sel["oiii_err"]/df_sel["oiii"])**2 + (df_sel["hb_err"]/df_sel["hb"])**2)

# BPT abscisse
df_sel["nii_ha"] = df_sel["nii"]/df_sel["ha"]
df_sel["sii_ha"] = df_sel["sii"]/df_sel["ha"]
df_sel["oi_ha"] = df_sel["oi"]/df_sel["ha"]

df_sel["nii_ha_err"] = df_sel["nii_ha"]*np.sqrt( (df_sel["nii_err"]/df_sel["nii"])**2 + (df_sel["ha_err"]/df_sel["ha"])**2)
df_sel["sii_ha_err"] = df_sel["sii_ha"]*np.sqrt( (df_sel["sii_err"]/df_sel["sii"])**2 + (df_sel["ha_err"]/df_sel["ha"])**2)
df_sel["oi_ha_err"] = df_sel["oi_ha"]*np.sqrt( (df_sel["oi_err"]/df_sel["oi"])**2 + (df_sel["ha_err"]/df_sel["ha"])**2)


In [ ]:
# Create figure
fig, ax = plt.subplots(figsize=(6,6))

# Kewley+ Line
x = np.logspace(-1.5,0.05,100)
y = 10**(0.61/(np.log10(x) - 0.05) + 1.3)
ax.plot(x,y,color='gray',ls='--',label='Kauffman+03')

# Kauffman+ Line
x = np.logspace(-1.5,0.47,100)
y = 10**(0.61/(np.log10(x) - 0.47) + 1.18)
ax.plot(x,y,color='gray',ls='-',label='Kewley+01')

# Plot BPT
nii_ha = df_sel["nii_ha"].values
oiii_hb = df_sel["oiii_hb"].values
nii_ha_err = df_sel["nii_ha_err"].values
oiii_hb_err = df_sel["oiii_hb_err"].values
ax.scatter(nii_ha,oiii_hb,color='k',label='Fors2',edgecolors='none',alpha=1.0)
#x,y,xerr,yerr = np.median(nii/ha),np.median(oiii/hb),np.std(nii/ha),np.std(oiii/hb)
#ax.errorbar(nii_ha,oiii_hb,xerr=nii_ha_err,yerr=oiii_hb_err,fmt=".",color='r',label='Average')
#ax.legend()

# Axis limits
ax.set(xlim=[1e-1,1e1],ylim=[1e-1,2e1])

# Axis labels
ax.set(xlabel=r'[NII]/H$\alpha$',ylabel=r'[OIII]/H$\beta$')

# Axis scale
ax.set(yscale='log',xscale='log')

# Show figure
ax.set_title("BPT for Fors2")

ax.legend()

In [ ]:
# Create figure
fig, (ax1,ax2,ax3) = plt.subplots(1,3,figsize=(18,6),sharey=True)

# Kewley+ Line
x = np.logspace(-1.5,0.05,100)
y = 10**(0.61/(np.log10(x) - 0.05) + 1.3)
ax1.plot(x,y,color='gray',ls='--',label='Kauffman+03')

# Kauffman+ Line
x = np.logspace(-1.5,0.47,100)
y = 10**(0.61/(np.log10(x) - 0.47) + 1.18)
ax1.plot(x,y,color='gray',ls='-',label='Kewley+01')

# Plot BPT 1
nii_ha = df_sel["nii_ha"].values
oiii_hb = df_sel["oiii_hb"].values
nii_ha_err = df_sel["nii_ha_err"].values
oiii_hb_err = df_sel["oiii_hb_err"].values
ax1.scatter(nii_ha,oiii_hb,color='k',label='Fors2',edgecolors='none',alpha=1.0)
#x,y,xerr,yerr = np.median(nii/ha),np.median(oiii/hb),np.std(nii/ha),np.std(oiii/hb)
#ax.errorbar(nii_ha,oiii_hb,xerr=nii_ha_err,yerr=oiii_hb_err,fmt=".",color='r',label='Average')
#ax.legend()

# Axis limits
ax1.set(xlim=[1e-1,1e1],ylim=[1e-1,2e1])

# Axis labels
ax1.set(xlabel=r'[NII]/H$\alpha$',ylabel=r'[OIII]/H$\beta$')

# Axis scale
ax1.set(yscale='log',xscale='log')

# Show figure
ax1.set_title("BPT for Fors2")
ax1.legend()


# Plot BPT 2
sii_ha = df_sel["sii_ha"].values
ax2.scatter(sii_ha,oiii_hb,color='k',label='Fors2',edgecolors='none',alpha=1.0)

# Axis labels
ax2.set(xlabel=r'[SII]/H$\alpha$',ylabel=r'[OIII]/H$\beta$')
# Axis scale
ax2.set(yscale='log',xscale='log')


# Plot BPT 3
oi_ha = df_sel["oi_ha"].values
ax3.scatter(oi_ha,oiii_hb,color='k',label='Fors2',edgecolors='none',alpha=1.0)

# Axis labels
ax3.set(xlabel=r'[OI]/H$\alpha$',ylabel=r'[OIII]/H$\beta$')
# Axis scale
ax3.set(yscale='log',xscale='log')

plt.tight_layout()